In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q pydicom pillow nltk sacrebleu tqdm

In [1]:
# Cell 1 — Install DICOM dependencies
import subprocess
subprocess.run(['pip', 'install', 'pydicom', 'pylibjpeg', 'pylibjpeg-libjpeg', '-q'])
print('Done.')

Done.


In [2]:
# Cell 2 — Imports and device
import os, json, re, string
import torch
import numpy as np
import pydicom
import pydicom.multival
from PIL import Image
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sacrebleu.metrics import BLEU
from tqdm import tqdm
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Adjusted to ensure CUDA is picked up on Kaggle T4 GPUs
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [3]:
# Cell 3 — DICOM image loading utilities
def apply_windowing(pixel_array: np.ndarray,
                    window_center: float,
                    window_width: float) -> np.ndarray:
    low  = window_center - window_width / 2
    high = window_center + window_width / 2
    arr  = pixel_array.astype(np.float32)
    arr  = np.clip(arr, low, high)
    arr  = (arr - low) / (high - low) * 255.0
    return arr.astype(np.uint8)

def dicom_to_pil(dcm_path: str) -> Image.Image:
    dcm   = pydicom.dcmread(dcm_path)
    array = dcm.pixel_array

    wc = getattr(dcm, 'WindowCenter', None)
    ww = getattr(dcm, 'WindowWidth',  None)

    if wc is None or ww is None:
        arr = array.astype(np.float32)
        arr = (arr - arr.min()) / max(arr.max() - arr.min(), 1) * 255
        return Image.fromarray(arr.astype(np.uint8)).convert('RGB')

    if isinstance(wc, pydicom.multival.MultiValue): wc = float(wc[0])
    if isinstance(ww, pydicom.multival.MultiValue): ww = float(ww[0])

    windowed = apply_windowing(array, float(wc), float(ww))
    return Image.fromarray(windowed).convert('RGB')

def get_series_representative_slices(dicom_root: str, study_id: str, series_id: str) -> list[str]:
    study_path = os.path.join(dicom_root, study_id)

    series_folder = None
    for folder in os.listdir(study_path):
        if folder.endswith(f'.{series_id}') and os.path.isdir(os.path.join(study_path, folder)):
            series_folder = os.path.join(study_path, folder)
            break

    if series_folder is None:
        raise FileNotFoundError(f'Series {series_id} not found in study {study_id}')

    dcm_files = sorted(
        [f for f in os.listdir(series_folder) if f.endswith('.dcm')],
        key=lambda f: int(f.split('.')[-3]) 
    )

    if not dcm_files:
        raise FileNotFoundError(f'No DICOM files in series folder: {series_folder}')

    n = len(dcm_files)
    indices = [int(n * p) for p in [0.2, 0.4, 0.6, 0.8]]
    indices = [max(0, min(n - 1, idx)) for idx in indices]
    final_indices = []
    for idx in indices:
        if idx not in final_indices:
            final_indices.append(idx)
    return [os.path.join(series_folder, dcm_files[i]) for i in final_indices]

def create_image_grid(images: list[Image.Image]) -> Image.Image:
    if not images:
        raise ValueError('No images to tile')
    w, h = images[0].size
    grid = Image.new('RGB', (w * 2, h * 2), (0, 0, 0))
    positions = [(0, 0), (w, 0), (0, h), (w, h)]
    for i, img in enumerate(images[:4]):
        grid.paste(img, positions[i])
    return grid

print('DICOM utilities defined.')

DICOM utilities defined.


In [4]:
# Cell 4 — Verify DICOM loading on all QA pairs
DICOM_ROOT   = '/kaggle/input/datasets/shriyanshraj/dicom-files/dicom_files'
DATASET_PATH = '/kaggle/input/datasets/shriyanshraj/clinical-vqa-dataset/clinical_vqa_dataset.jsonl'

records = [json.loads(l) for l in open(DATASET_PATH)]

print(f'Loaded {len(records)} QA pairs\n')
print(f'{"#":<3} {"Study":<8} {"Series":<8} {"Type":<8} {"Question":<55} {"Answer"}')
print('-' * 110)

errors = []
for i, r in enumerate(records):
    study_short = r['study_id'][-8:]
    try:
        paths = get_series_representative_slices(DICOM_ROOT, r['study_id'], r['target_series'])
        imgs = [dicom_to_pil(p) for p in paths]
        img  = create_image_grid(imgs)
        status = f'OK {img.size}'
    except Exception as e:
        status = f'ERROR: {e}'
        errors.append(i)

    print(f'{i:<3} ...{study_short:<8} S{r["target_series"]:<7} {r["answer_type"]:<8} '
          f'{r["question"][:55]:<55} {r["answer"][:20]}')
    print(f'    Image: {status}')

print(f'\nVerification complete. {len(records) - len(errors)}/{len(records)} images loaded OK.')
if errors:
    print(f'Errors at indices: {errors}')

Loaded 16 QA pairs

#   Study    Series   Type     Question                                                Answer
--------------------------------------------------------------------------------------------------------------
0   ...31550463 S6       CLOSED   Is there evidence of a medial meniscus tear in this sca No
    Image: OK (320, 320)
1   ...31550463 S8       CLOSED   Are the anterior and posterior cruciate ligaments intac Yes
    Image: OK (320, 320)
2   ...31550463 S7       OPEN     What is the primary abnormal fluid finding in this join Small effusion
    Image: OK (320, 320)
3   ...31550463 S3       CLOSED   Is a Baker's cyst evident in the popliteal region?      No
    Image: OK (320, 320)
4   ...31550463 S4       OPEN     What is the condition of the medial and lateral collate Intact
    Image: OK (320, 320)
5   ...31550463 S7       CLOSED   Is there any evidence of a bone bruise or fracture?     No
    Image: OK (320, 320)
6   ...31550463 S3       CLOSED   Is chondromalaci

In [5]:
# Cell 5 — Visualise representative slices for all QA pairs
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

seen = set()
panels = []
for r in records:
    key = (r['study_id'], r['target_series'])
    if key not in seen:
        seen.add(key)
        try:
            paths = get_series_representative_slices(DICOM_ROOT, r['study_id'], r['target_series'])
            dcm  = pydicom.dcmread(path)
            imgs = [dicom_to_pil(p) for p in paths]
        img  = create_image_grid(imgs)
            desc = getattr(dcm, 'SeriesDescription', f'Series {r["target_series"]}')
            patient = 'P1' if '524' in r['study_id'] else 'P2'
            panels.append((img, f'{patient} S{r["target_series"]}\n{desc}'))
        except Exception as e:
            print(f'Warning: {e}')

n = len(panels)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4 * ((n + 1) // 2), 8))
axes = axes.flatten()

for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=8)
    ax.axis('off')

for ax in axes[len(panels):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('clinical_vqa_images.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved clinical_vqa_images.png — {n} unique series shown')

Saved clinical_vqa_images.png — 9 unique series shown


In [6]:
# Cell 6 — Scoring functions
stemmer     = PorterStemmer()
STOPWORDS   = {'the','a','an','is','are','was','were','this','that','of','in','for'}
bleu_metric = BLEU(effective_order=True)

def tokenize_answer(text: str) -> list:
    text = re.sub(r'\*+', '', text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def normalize_answer(text: str) -> str:
    text = re.sub(r'\*+', '', text)
    text = re.split(r'(?<=[.!?])\s', text)[0]
    text = text.replace('/', ' ').lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in STOPWORDS]
    return ' '.join(tokens)

def token_f1(prediction: str, ground_truth: str) -> dict:
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

def token_f1_normalized(prediction: str, ground_truth: str) -> dict:
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens   = normalize_answer(ground_truth).split()
    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

def is_correct(prediction: str, ground_truth: str, is_closed: bool) -> bool:
    scores    = token_f1(prediction, ground_truth)
    threshold = 0.5 if is_closed else 0.75
    return scores['recall'] >= threshold

def compute_bleu(prediction: str, ground_truth: str) -> float:
    return bleu_metric.sentence_score(
        hypothesis=prediction.lower(),
        references=[ground_truth.lower()]
    ).score

def score_results(results: list) -> dict:
    closed = [r for r in results if r['is_closed']]
    open_  = [r for r in results if not r['is_closed']]

    def avg_f1(recs, fn=token_f1):
        if not recs: return 0.0
        return sum(fn(r['prediction'], r['ground_truth'])['f1'] for r in recs) / len(recs)
    def accuracy(recs, closed, fn=token_f1):
        if not recs: return 0.0
        threshold = 0.5 if closed else 0.75
        return sum(1 for r in recs if fn(r['prediction'], r['ground_truth'])['recall'] >= threshold) / len(recs)
    def avg_bleu(recs):
        if not recs: return 0.0
        return sum(compute_bleu(r['prediction'], r['ground_truth']) for r in recs) / len(recs)

    return {
        'n_total':         len(results),
        'n_closed':        len(closed),
        'n_open':          len(open_),
        'overall_f1':      round(avg_f1(results) * 100, 2),
        'overall_f1_norm': round(avg_f1(results, token_f1_normalized) * 100, 2),
        'closed_acc':      round(accuracy(closed, True)  * 100, 2),
        'open_acc':        round(accuracy(open_,  False) * 100, 2),
        'bleu':            round(avg_bleu(results), 2),
    }

print('Scoring functions defined.')

Scoring functions defined.


In [7]:
# Cell 7 — Prompt builders
def build_prompt_clinical(question: str, is_closed: bool) -> str:
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return (
        f'{prefix}This is a composite image showing four MRI slices of a knee joint arranged in a 2x2 grid, sampled at 20%, 40%, 60%, and 80% depth through the series. Evaluate all four panels together when answering. '
        f'{question} '
        f"You may write out your argument before stating your final very short, "
        f"definitive, and concise answer (if possible, a single word or short phrase) "
        f"X in the format 'Final Answer: X'"
    )

def extract_final_answer(text: str) -> str:
    clean = re.sub(r'\*+', '', text).strip()
    match = re.search(r'[Ff]inal\s+[Aa]nswer\s*:\s*(.+)', clean, re.DOTALL)
    if match:
        answer = match.group(1).strip()
        answer = answer.split('\n')[0].strip()
        answer = answer.strip(string.punctuation + ' ')
        return answer
    return re.split(r'(?<=[.!?])\s', clean)[0].strip()

print('Closed prompt:')
print(build_prompt_clinical('Is there evidence of a meniscal tear?', is_closed=True))
print('\nOpen prompt:')
print(build_prompt_clinical('Where is the most pronounced cartilage loss?', is_closed=False))

Closed prompt:
Answer the question with yes or no. This is an MRI image of a knee joint. Is there evidence of a meniscal tear? You may write out your argument before stating your final very short, definitive, and concise answer (if possible, a single word or short phrase) X in the format 'Final Answer: X'

Open prompt:
This is an MRI image of a knee joint. Where is the most pronounced cartilage loss? You may write out your argument before stating your final very short, definitive, and concise answer (if possible, a single word or short phrase) X in the format 'Final Answer: X'


In [8]:
# Hugging-face-login
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret('HF_TOKEN'))
print('Logged in.')


Logged in.


In [9]:
# Cell 8 — Load model
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = 'google/medgemma-4b-it'
# MODEL_NAME = 'google/gemma-3-4b-it'

print(f'Loading {MODEL_NAME}...')
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model     = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='cuda:0' if device == 'cuda' else None 
)

if device != 'cuda':
    model = model.to(device)
    
model.eval()
print(f'Model loaded on {next(model.parameters()).device}')

Loading google/medgemma-4b-it...


The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model loaded on cuda:0


In [10]:
# Cell 9 — Inference function
def run_inference_clinical(image: Image.Image,
                           question: str,
                           is_closed: bool) -> tuple[str, str]:
    prompt_text = build_prompt_clinical(question, is_closed)
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': prompt_text},
        ]
    }]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, images=image, return_tensors='pt').to(device)
    
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=1.0,
        )
    input_len = inputs['input_ids'].shape[-1]
    raw       = processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()
    return extract_final_answer(raw), raw

print('Inference function defined.')

Inference function defined.


In [11]:
# Cell 10 — Smoke test
print('=== SMOKE TEST: 3 questions ===\n')

for r in records[:3]:
    is_closed = r['answer_type'] == 'CLOSED'
    try:
        paths = get_series_representative_slices(DICOM_ROOT, r['study_id'], r['target_series'])
        imgs = [dicom_to_pil(p) for p in paths]
        img  = create_image_grid(imgs)
        pred, raw = run_inference_clinical(img, r['question'], is_closed)
        f1 = token_f1(pred, r['answer'])['f1']
        print(f'Q:    {r["question"]}')
        print(f'GT:   {r["answer"]}')
        print(f'Pred: {pred}')
        print(f'F1:   {f1:.3f}')
        print(f'Raw:  {raw[:120]}')
        print()
    except Exception as e:
        print(f'Error: {e}\n')

=== SMOKE TEST: 3 questions ===

Q:    Is there evidence of a medial meniscus tear in this scan?
GT:   No
Pred: No
F1:   1.000
Raw:  The image shows a cross-sectional view of the knee joint. While it's difficult to definitively diagnose a meniscus tear 

Q:    Are the anterior and posterior cruciate ligaments intact?
GT:   Yes
Pred: Yes
F1:   1.000
Raw:  Based on the MRI image, it is difficult to definitively determine the integrity of the anterior and posterior cruciate l

Q:    What is the primary abnormal fluid finding in this joint?
GT:   Small effusion
Pred: Meniscal tear
F1:   0.000
Raw:  The image shows a knee joint. The primary abnormal fluid finding appears to be a **meniscal tear**.

Final Answer: Menis



In [12]:
# Cell 11 — Full evaluation run with checkpoint saving
OUTPUT_DIR  = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
safe_model  = MODEL_NAME.replace('/', '_')
output_path = os.path.join(OUTPUT_DIR, f'{safe_model}__clinical_knee_mri_v2.jsonl')

completed = {}
if os.path.exists(output_path):
    for line in open(output_path):
        r = json.loads(line)
        completed[r['idx']] = r
    print(f'Resuming: {len(completed)} already done.')

results = list(completed.values())
f_out   = open(output_path, 'a')
errors  = 0

for i, r in enumerate(tqdm(records, desc=f'{MODEL_NAME} | clinical_knee_mri')):
    if i in completed:
        continue
    is_closed = r['answer_type'] == 'CLOSED'
    try:
        paths     = get_series_representative_slices(DICOM_ROOT, r['study_id'], r['target_series'])
        imgs      = [dicom_to_pil(p) for p in paths]
        img       = create_image_grid(imgs)
        pred, raw = run_inference_clinical(img, r['question'], is_closed)
        record    = {
            'idx':          i,
            'study_id':     r['study_id'],
            'series':       r['target_series'],
            'question':     r['question'],
            'ground_truth': r['answer'],
            'prediction':   pred,
            'raw_output':   raw,
            'is_closed':    is_closed,
            'model':        MODEL_NAME,
            'dataset':      'clinical_knee_mri',
        }
    except Exception as e:
        errors += 1
        record = {
            'idx': i, 'study_id': r['study_id'], 'series': r['target_series'],
            'question': r['question'], 'ground_truth': r['answer'],
            'prediction': '', 'raw_output': '', 'is_closed': is_closed,
            'model': MODEL_NAME, 'dataset': 'clinical_knee_mri',
            'error': str(e),
        }

    results.append(record)
    f_out.write(json.dumps(record) + '\n')
    f_out.flush()

f_out.close()
print(f'\nDone. {len(results)} results ({errors} errors) -> {output_path}')

Resuming: 16 already done.


google/medgemma-4b-it | clinical_knee_mri: 100%|██████████| 16/16 [00:00<00:00, 143702.06it/s]


Done. 16 results (0 errors) -> ../outputs/google_medgemma-4b-it__clinical_knee_mri_v2.jsonl


In [13]:
# Cell 12 — Score results
results = [json.loads(l) for l in open(output_path)]
results = [r for r in results if 'error' not in r]

scores = score_results(results)

print(f'=== {MODEL_NAME} | Clinical Knee MRI ===\n')
print(f'Total questions:  {scores["n_total"]}')
print(f'  Closed (Y/N):   {scores["n_closed"]}')
print(f'  Open-ended:     {scores["n_open"]}')
print()
print(f'Overall F1:       {scores["overall_f1"]}%')
print(f'Overall F1 Norm:  {scores["overall_f1_norm"]}%')
print(f'Closed Accuracy:  {scores["closed_acc"]}%')
print(f'Open Accuracy:    {scores["open_acc"]}%')
print(f'BLEU:             {scores["bleu"]}')
print()

print('Per-question breakdown:')
print(f'{"#":<3} {"Type":<8} {"GT":<30} {"Pred":<30} {"F1":>6}')
print('-' * 82)
for r in sorted(results, key=lambda x: x['idx']):
    f1  = token_f1(r['prediction'], r['ground_truth'])['f1']
    typ = 'CLOSED' if r['is_closed'] else 'OPEN'
    print(f'{r["idx"]:<3} {typ:<8} {r["ground_truth"]:<30} {r["prediction"][:28]:<30} {f1:>6.3f}')

=== google/medgemma-4b-it | Clinical Knee MRI ===

Total questions:  16
  Closed (Y/N):   9
  Open-ended:     7

Overall F1:       60.42%
Overall F1 Norm:  60.42%
Closed Accuracy:  88.89%
Open Accuracy:    14.29%
BLEU:             59.69

Per-question breakdown:
#   Type     GT                             Pred                               F1
----------------------------------------------------------------------------------
0   CLOSED   No                             No                              1.000
1   CLOSED   Yes                            Yes                             1.000
2   OPEN     Small effusion                 Meniscal tear                   0.000
3   CLOSED   No                             Yes                             0.000
4   OPEN     Intact                         Intact                          1.000
5   CLOSED   No                             No                              1.000
6   CLOSED   No                             No                              1.000

In [14]:
# Cell 13 — Patient-level breakdown
from collections import defaultdict

by_patient = defaultdict(list)
for r in results:
    by_patient[r['study_id']].append(r)

print('=== Per-patient scores ===\n')
for study_id, patient_results in by_patient.items():
    s       = score_results(patient_results)
    patient = 'Patient 1' if '524' in study_id else 'Patient 2'
    print(f'{patient} ({study_id[-8:]})')
    print(f'  Questions: {s["n_total"]} ({s["n_closed"]} closed, {s["n_open"]} open)')
    print(f'  F1: {s["overall_f1"]}%  |  Norm F1: {s["overall_f1_norm"]}%  |  '
          f'Closed Acc: {s["closed_acc"]}%  |  Open Acc: {s["open_acc"]}%')
    print()

=== Per-patient scores ===

Patient 1 (31550463)
  Questions: 8 (5 closed, 3 open)
  F1: 62.5%  |  Norm F1: 62.5%  |  Closed Acc: 80.0%  |  Open Acc: 33.33%

Patient 2 (67111743)
  Questions: 8 (4 closed, 4 open)
  F1: 58.33%  |  Norm F1: 58.33%  |  Closed Acc: 100.0%  |  Open Acc: 0.0%



In [15]:
import gc
import torch
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Delete all possible tensor references that might be lingering in your local scope
try:
    del model, processor, inputs, output_ids, text, raw, img
except NameError:
    pass

# 2. The Magic Bullet: Clear Jupyter's hidden execution history
# Jupyter stores every output. This forces it to forget them.
%reset -f Out 

# 3. Triple-tap the garbage collector
gc.collect()
gc.collect()
gc.collect()

# 4. Empty cache and collect Inter-Process Communication memory
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# 5. Print a quick memory summary to verify it worked
allocated = torch.cuda.memory_allocated() / (1024 ** 2)
reserved = torch.cuda.memory_reserved() / (1024 ** 2)
print(f"Memory Allocated: {allocated:.2f} MB")
print(f"Memory Reserved:  {reserved:.2f} MB")
print("If these are near 0 MB, you are safe to load Gemma 3!")

Flushing output cache (0 entries)
Memory Allocated: 9.12 MB
Memory Reserved:  22.00 MB
If these are near 0 MB, you are safe to load Gemma 3!


In [16]:
!ls -lah ../outputs/

total 16K
drwxr-xr-x 2 root root 4.0K Jun 16 09:22 .
drwxr-xr-x 6 root root 4.0K Jun 16 09:22 ..
-rw-r--r-- 1 root root 7.7K Jun 16 09:24 google_medgemma-4b-it__clinical_knee_mri_v2.jsonl


In [17]:
# Cell 14 — Load model gemma 3
from transformers import AutoProcessor, AutoModelForImageTextToText

# MODEL_NAME = 'google/medgemma-4b-it'
MODEL_NAME = 'google/gemma-3-4b-it'

print(f'Loading {MODEL_NAME}...')
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model     = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='cuda:0' if device == 'cuda' else None 
)

if device != 'cuda':
    model = model.to(device)
    
model.eval()
print(f'Model loaded on {next(model.parameters()).device}')

Loading google/gemma-3-4b-it...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded on cuda:0


In [18]:
# Cell 15 — Full evaluation run with checkpoint saving for gemma 3
OUTPUT_DIR  = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
safe_model  = MODEL_NAME.replace('/', '_')
output_path = os.path.join(OUTPUT_DIR, f'{safe_model}__clinical_knee_mri_v2.jsonl')

completed = {}
if os.path.exists(output_path):
    for line in open(output_path):
        r = json.loads(line)
        completed[r['idx']] = r
    print(f'Resuming: {len(completed)} already done.')

results = list(completed.values())
f_out   = open(output_path, 'a')
errors  = 0

for i, r in enumerate(tqdm(records, desc=f'{MODEL_NAME} | clinical_knee_mri')):
    if i in completed:
        continue
    is_closed = r['answer_type'] == 'CLOSED'
    try:
        paths     = get_series_representative_slices(DICOM_ROOT, r['study_id'], r['target_series'])
        imgs      = [dicom_to_pil(p) for p in paths]
        img       = create_image_grid(imgs)
        pred, raw = run_inference_clinical(img, r['question'], is_closed)
        record    = {
            'idx':          i,
            'study_id':     r['study_id'],
            'series':       r['target_series'],
            'question':     r['question'],
            'ground_truth': r['answer'],
            'prediction':   pred,
            'raw_output':   raw,
            'is_closed':    is_closed,
            'model':        MODEL_NAME,
            'dataset':      'clinical_knee_mri',
        }
    except Exception as e:
        errors += 1
        record = {
            'idx': i, 'study_id': r['study_id'], 'series': r['target_series'],
            'question': r['question'], 'ground_truth': r['answer'],
            'prediction': '', 'raw_output': '', 'is_closed': is_closed,
            'model': MODEL_NAME, 'dataset': 'clinical_knee_mri',
            'error': str(e),
        }

    results.append(record)
    f_out.write(json.dumps(record) + '\n')
    f_out.flush()

f_out.close()
print(f'\nDone. {len(results)} results ({errors} errors) -> {output_path}')

google/gemma-3-4b-it | clinical_knee_mri:   0%|          | 0/16 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
google/gemma-3-4b-it | clinical_knee_mri: 100%|██████████| 16/16 [02:36<00:00,  9.81s/it]


Done. 16 results (0 errors) -> ../outputs/google_gemma-3-4b-it__clinical_knee_mri_v2.jsonl


In [19]:
# Cell 16 — Score results for gemma 3
results = [json.loads(l) for l in open(output_path)]
results = [r for r in results if 'error' not in r]

scores = score_results(results)

print(f'=== {MODEL_NAME} | Clinical Knee MRI ===\n')
print(f'Total questions:  {scores["n_total"]}')
print(f'  Closed (Y/N):   {scores["n_closed"]}')
print(f'  Open-ended:     {scores["n_open"]}')
print()
print(f'Overall F1:       {scores["overall_f1"]}%')
print(f'Overall F1 Norm:  {scores["overall_f1_norm"]}%')
print(f'Closed Accuracy:  {scores["closed_acc"]}%')
print(f'Open Accuracy:    {scores["open_acc"]}%')
print(f'BLEU:             {scores["bleu"]}')
print()

print('Per-question breakdown:')
print(f'{"#":<3} {"Type":<8} {"GT":<30} {"Pred":<30} {"F1":>6}')
print('-' * 82)
for r in sorted(results, key=lambda x: x['idx']):
    f1  = token_f1(r['prediction'], r['ground_truth'])['f1']
    typ = 'CLOSED' if r['is_closed'] else 'OPEN'
    print(f'{r["idx"]:<3} {typ:<8} {r["ground_truth"]:<30} {r["prediction"][:28]:<30} {f1:>6.3f}')

=== google/gemma-3-4b-it | Clinical Knee MRI ===

Total questions:  16
  Closed (Y/N):   9
  Open-ended:     7

Overall F1:       4.17%
Overall F1 Norm:  4.17%
Closed Accuracy:  0.0%
Open Accuracy:    0.0%
BLEU:             2.3

Per-question breakdown:
#   Type     GT                             Pred                               F1
----------------------------------------------------------------------------------
0   CLOSED   No                             Yes                             0.000
1   CLOSED   Yes                            No                              0.000
2   OPEN     Small effusion                 Effusion                        0.667
3   CLOSED   No                             Yes                             0.000
4   OPEN     Intact                         Partial MCL tear                0.000
5   CLOSED   No                             Yes                             0.000
6   CLOSED   No                             Yes                             0.000
7   OPEN

In [20]:
# Cell 17 — Patient-level breakdown for gemma 3
from collections import defaultdict

by_patient = defaultdict(list)
for r in results:
    by_patient[r['study_id']].append(r)

print('=== Per-patient scores ===\n')
for study_id, patient_results in by_patient.items():
    s       = score_results(patient_results)
    patient = 'Patient 1' if '524' in study_id else 'Patient 2'
    print(f'{patient} ({study_id[-8:]})')
    print(f'  Questions: {s["n_total"]} ({s["n_closed"]} closed, {s["n_open"]} open)')
    print(f'  F1: {s["overall_f1"]}%  |  Norm F1: {s["overall_f1_norm"]}%  |  '
          f'Closed Acc: {s["closed_acc"]}%  |  Open Acc: {s["open_acc"]}%')
    print()

=== Per-patient scores ===

Patient 1 (31550463)
  Questions: 8 (5 closed, 3 open)
  F1: 8.33%  |  Norm F1: 8.33%  |  Closed Acc: 0.0%  |  Open Acc: 0.0%

Patient 2 (67111743)
  Questions: 8 (4 closed, 4 open)
  F1: 0.0%  |  Norm F1: 0.0%  |  Closed Acc: 0.0%  |  Open Acc: 0.0%

